# HAR Transformer — Google Colab Setup

**Checklist before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `har_transformer/` folder (or clone from your repo)
3. Upload the UCI HAR Dataset zip
4. Run cells top-to-bottom


In [1]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU')

: 

In [1]:
# ── Cell 2: Install dependencies ────────────────────────────────────
# TF 2.x + sklearn are pre-installed on Colab; only seaborn may need upgrading
!pip install -q seaborn --upgrade

In [ ]:
# ── Cell 3: Download UCI HAR Dataset ────────────────────────────────
# Option A: wget directly from UCI repository
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip -O UCI_HAR.zip
!unzip -q UCI_HAR.zip
!ls 'UCI HAR Dataset/'

In [ ]:
# ── Cell 4: Upload project files ────────────────────────────────────
# If you have the har_transformer/ folder as a zip, upload it here:
# from google.colab import files
# uploaded = files.upload()   # upload har_transformer.zip
# !unzip -q har_transformer.zip

# --- OR clone from GitHub ---
# !git clone https://github.com/YOUR_USER/har_transformer.git

# Verify structure
!ls har_transformer/

In [ ]:
# ── Cell 5: Move dataset to project folder ───────────────────────────
# The dataset loader expects 'UCI HAR Dataset' inside the project folder
!cp -r 'UCI HAR Dataset' har_transformer/
!ls har_transformer/

In [ ]:
# ── Cell 6: Change to project directory and run ──────────────────────
import os
os.chdir('har_transformer')
print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 7: Quick sanity check ───────────────────────────────────────
from dataset import load_dataset, normalize
X_train, y_train, X_test, y_test = load_dataset('UCI HAR Dataset')
X_train, X_test = normalize(X_train, X_test)
print('X_train:', X_train.shape, '  y_train:', y_train.shape)
print('X_test :', X_test.shape,  '  y_test :', y_test.shape)

In [ ]:
# ── Cell 8: Model architecture summaries ────────────────────────────
from model import MODEL_REGISTRY, build_model
for name in MODEL_REGISTRY:
    m = build_model(name)
    print(f'\n--- {name.upper()} ---')
    m.summary()

In [ ]:
# ── Cell 9: Run the FULL pipeline ───────────────────────────────────
# This trains all 3 models, runs 5-fold CV, evaluates, and saves all figures.
# Expected wall-clock time on T4 GPU: ~25-40 minutes total.
%run run_all.py

In [ ]:
# ── Cell 10: Launch TensorBoard ─────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir logs/

In [ ]:
# ── Cell 11: Display saved figures inline ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

for png in sorted(glob.glob('figures/*.png')):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(mpimg.imread(png))
    ax.axis('off')
    ax.set_title(png)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 12: Download all outputs as a zip ───────────────────────────
import shutil
from google.colab import files

# Bundle figures + tables
shutil.make_archive('har_outputs', 'zip', '.', 'figures')
shutil.make_archive('har_tables',  'zip', '.', 'tables')
files.download('har_outputs.zip')
files.download('har_tables.zip')